### Building a Project with Groq free API Key

In [1]:
import requests
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Initialize Google client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# FUNCTION 1 - Get country data

def get_country_data(country_name):
    try:
        response = requests.get(
            f"https://restcountries.com/v3.1/name/{country_name}", timeout=5
        )
        response.raise_for_status()

        country = response.json()[0]

        return {
            "name": country.get("name", {}).get("common", "Unknown"),
            "capital": country.get("capital", ["Unknown"])[0],
            "population": country.get("population", "Unknown"),
            "region": country.get("region", "Unknown"),
            "currency": list(country.get("currencies", {}).keys())[0] if country.get("currencies")else "Unknown",
            "latitude": country.get("latlng", [0])[0],
            "longitude": country.get("latlng", [0, 0])[1]
        }

    except requests.exceptions.ConnectionError:
        print("No Internet Connection")
        return None
    except requests.exceptions.Timeout:
        print("Request timed out")
        return None
    except requests.exceptions.HTTPError:
        print(f"HTTP error: '{country_name}' not found" )
        return None

get_country_data("indonesia")

# FUNCTION 2 - Get weather data

def get_weather(latitude, longitude):
    try:
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "current_weather": True,
            "timezone": "auto"
        }

        response = requests.get(
             "https://api.open-meteo.com/v1/forecast", params=params, timeout=5
        )
        response.raise_for_status()

        weather = response.json().get("current_weather", {})

        return {
            "temperature": weather.get("temperature", "Unknown"),
            "windspeed": weather.get("windspeed", "Unknown")
        }

    except requests.exceptions.RequestException as e:
        print("Weather fetch failed", e)
        return None

get_weather(-5.0, 120.0)

# FUNCTION 3 - Generate AI travel summary

def generate_travel_summary(country_data, weather_data):
    prompt = f"""
    Here is some informeation about {country_data['name']}:

    - Capital city: {country_data['capital']}
    - Population: {country_data['population']}
    - Region: {country_data['region']}
    - Currency: {country_data['currency']}
    - Current temperature: {weather_data['temperature']}°C
    - Curent windspeed: {weather_data['windspeed']} km/h

    Write a short, fun, and engaging travel summary about this country in 3-4 sentences. Make it sound exciting for a potential traveler.
    """

    response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
    return response.choices[0].message.content

# MAIN PROGRAM

def main():
    print("🌍 AI Country Explorer")
    print("=" * 40)

    country_name = input("Enter a country name: ")

    print(f"\nFetching data for {country_name}...")
    country_data = get_country_data(country_name)

    if not country_data:
        return

    print("Fetching weather data...")
    weather_data = get_weather(
        country_data["latitude"],
        country_data["longitude"]
    )

    if not weather_data:
        return

    print("Generating AI travel summary...")
    summary = generate_travel_summary(country_data, weather_data)

    #Print Final results
    print("\n" + "=" * 40)
    print(f"🏳️ Country: {country_data['name']}"),
    print(f"Capital: {country_data['capital']}"),
    print(f"👥 Ppulation: {country_data['population']:,}"),
    print(f"Region: {country_data['region']}"),
    print(f"💰 Currency: {country_data['currency']}")
    print(f"🌡️ Temperature: {weather_data['temperature']}°C")
    print(f"Wind speed: {weather_data['windspeed']}km/h")
    print("\n🤖 AI Travel Summary:")
    print(summary)

main()

🌍 AI Country Explorer

Fetching data for japan...
Fetching weather data...
Generating AI travel summary...

🏳️ Country: Japan
Capital: Tokyo
👥 Ppulation: 123,210,000
Region: Asia
💰 Currency: JPY
🌡️ Temperature: 17.3°C
Wind speed: 4.7km/h

🤖 AI Travel Summary:
Get ready for the adventure of a lifetime in Japan, a vibrant country located in the heart of Asia. With its bustling capital city Tokyo, rich culture, and breathtaking landscapes, Japan is a destination that will leave you in awe. As you step foot in this incredible country, you'll be greeted by mild temperatures of around 17°C and gentle breezes of 4.7 km/h, setting the perfect tone for an unforgettable journey. From vibrant cities to serene landscapes, Japan is a traveler's paradise waiting to be explored!
